# Quickstart

## Your First Topology

As in Kafka Streams, in Kafi Streams, a processing pipeline is called *topology*.

A Kafi Streams topology is a directed acyclic graph of operators starting with arbitrary many *sources* and ending with arbitrary many *sinks* (typically corresponding to Kafka topics).

Here is a concrete example, displayed in the traditional Kafka Streams-like way (see https://zz85.github.io/kafka-streams-viz/):

```mermaid
graph TD
25aa7736-238a-40f0-8054-d9c6f986ee3b[source_clicks] --> a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op]
fcb627f1-8b99-408c-941a-130bfbf828c4[map_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op] --> 56b14bac-535b-4bfa-81ae-6347726eb5f9[sink_joined]
14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op] --> 536f3bac-98f3-4b53-bb64-e3874777f1ce[join_equi_op]
a463ea6a-2d84-4e5d-bbf6-6d7cd7a22eb5[map_op] --> 14a0f926-3b42-4f6a-ad3a-0f0fc40138b7[filter_op]
2c812b6a-4b42-4293-a143-b36e797d0a84[source_customers] --> fcb627f1-8b99-408c-941a-130bfbf828c4[map_op]
```

There are two sources (`source_clicks` and `source_customers`) at the top. Each source goes through a `map` operaotor. The clicks, in addition, go through a `filter` operator. Then, both sides are joined by the `join_equi` operator and end up in the sink (`sink_joined`).



## The Data

We assume that `clicks` is a source of Kafka messages like this:

In [ ]:
{
    "key": None,
    "value": {"customer_id": "4711",
              "view_time": 200,
              "ts": 1609457200000},
    "partition": 2,
    "offset": 23,
    "timestamp": 1609457201000,
    "headers": None
}

...and that `customers` is a source of Kafka messages like this:

In [ ]:
{
    "key": "4711",
    "value": {"id": "4711",
              "name": "Sallyann Jupp"},
    "partition": 0,
    "offset": 67,
    "timestamp": 1609457001000,
    "headers": None
}

The aim of the topology is to join `clicks` with `customers` to enrich the `customer_id` in the `clicks` with the `name` of the customer from the `customers`. An example output message looks like this (only the `value` field of the Kafka message is relevant here):


In [ ]:
{"value": {"customer_id": "4711",
           "view_time": 200,
           "name": "Sallyann Jupp"}
}

So now that we know how the data that we are about to deal with looks like, we can already write generators for it that we will need a bit later on:

In [3]:
import random, time

from faker import Faker

customers_int = 100

class ClickGenerator:
    def __init__(self):
        self.ts_int = int(time.time() * 1000)
        self.ts_step_int = 100
        self.customer_id_int = 0

    def generate(self):
        message_dict = {
            "key": None,
            "value": {"customer_id": random.randint(0, customers_int - 1),
                      "view_time": random.randint(10, 120),
                      "ts": self.ts_int},
        }
        #
        self.ts_int += self.ts_step_int
        #
        return message_dict

class CustomerGenerator:
    def __init__(self):
        self.customer_id_int = 0
        self.customer_id_int_name_str_dict = {}
        fake = Faker()
        for customer_id_int in range(customers_int):
            name_str = fake.name()
            self.customer_id_int_name_str_dict[customer_id_int] = name_str

    def generate(self):
        customer_id_int = random.randint(0, customers_int - 1)
        message_dict = {
            "key": str(customer_id_int),
            "value": {"id": customer_id_int,
                      "name": self.customer_id_int_name_str_dict[customer_id_int]}
        }
        #
        return message_dict
    
click_generator = ClickGenerator()
for _ in range(3):
    print(click_generator.generate())

customer_generator = CustomerGenerator()
for _ in range(3):
    print(customer_generator.generate())


{'key': None, 'value': {'customer_id': 83, 'view_time': 23, 'ts': 1786463606074}}
{'key': None, 'value': {'customer_id': 44, 'view_time': 61, 'ts': 1786463606174}}
{'key': None, 'value': {'customer_id': 71, 'view_time': 65, 'ts': 1786463606274}}
{'key': '82', 'value': {'id': 82, 'name': 'Kelsey Church'}}
{'key': '62', 'value': {'id': 62, 'name': 'Daniel Mendoza'}}
{'key': '42', 'value': {'id': 42, 'name': 'Morgan Lowery'}}


## You First Topology - In Code

Now we can start writing the real thing. The topology.

In Kafi Streams, you specify topologies using a Kafka Streams DSL-inspired fluent API, i.e., in code:


In [4]:
# 1. Boilerplate

import sys
sys.path.insert(1, "..")

import importlib
import kafi.streams.topologynode
import kafi.streams.streams
importlib.reload(kafi.streams.topologynode)
importlib.reload(kafi.streams.streams)

from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.INFO)

# 2. Connect to Kafka

from kafi.kafka.cluster.cluster import Cluster
c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

# 3. Specify the Topology

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"

## a) Clicks

click_tn = (
    Streams.source(c, click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

## b) Customers

customer_tn = (
    Streams.source(c, customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

## c) Join and Sink

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "name": r_r["name"]}})
    .sink(c, sink_str)
)

# 4. Build the Topology

b_tn = Streams.build(sink_tn)


Let's go through the code in baby steps.

### 1. Boilerplate

We start with some boilerplate. We import `Streams` and set up the logging. `Streams` is a subclass of the class `TopologyNode`:
* `TopologyNode`: pydbsp-based stream processing; no dependency at all to Kafka.
* `Streams`: Subclass of `TopologyNode`; Kafi/Kafka wrapper around `TopologyNode`.

### 2. Connect to Kafka

We connect to Kafka. In the example, we connect to a locally installed Kafka cluster.

### 3. Specify the Topology

This is the most interested step: We specify our first Kafi Streams topology. The topology consists of three parts.

#### a) Clicks

The first part of the topology is about the clicks:
1. We define the source with `Streams.source()`. As for the parameters, `c` stands for the Kafka cluster and `click_source_str` for the topic name. As you can see, contrary to Kafka Streams, all sources and sinks can be on any Kafka cluster :)
2. We use the `map` operator to select two fields from the value of the incoming click events (`customer_id` and `view_time` from the `value`).
3. We employ the `filter` operator to filter out those incoming click events where `view_time` is smaller than `100`.

As for the naming conventions used throughout:
* `_tn` is a suffix for instances of the Kafi Streams `TopologyNode` class.
* `r` stands for "record" (=typically a Python dictionary); `_r` is the corresponding suffix.

#### b) Customers

The second part of the topology is about the customers:
1. We define the source.
2. We select two fields from the value of the incoming customer messages (`id` and `name`).

### c) Join and Sink

The third and last part of the topology joins the clicks and customers using an equi join (`join_equi`) and sinks the result.

The arguments of the `join_equi()` operator are:
1. Right side of the join (here: `customer_tn`)
2. Left side join key (here: the `customer_id` field of the left side = the clicks)
3. Right side join key (here: the `id` field of the right side = the customers)
4. Projection function. Here: `customer_id` and `view_time` from the clicks, and `name` from the customers.

At the end of the topology is the sink specification (cluster `c` and topic name `sink_str`).

### 4. Building the Topology

Before you can use a Kafi Streams topology, it needs to be "built". Under the covers, this creates a pydbsp "circuit" that is eventually used for the processing:


## Start the Processing

Now we are ready to rumble. Let's reset the topology, (re-)create the sources and the sink, and start the processing:

In [5]:
b_tn.reset()

c.recreate(click_source_str)
c.recreate(customer_source_str)
c.recreate(sink_str)

stop = Streams.start_streams(b_tn, progress=True)


(['customers'], 'streams_1786463668767')
(['clicks'], 'streams_1786463668767')


Now Kafi Streams is still idling around, there is no data coming...

So let's not forget to produce some data to the two input topics (10.000 messages to `clicks` and 10.000 to `customers`) and see what happens:

In [7]:
click_gen = ClickGenerator()
customer_gen = CustomerGenerator()

click_pr = c.producer(click_source_str)
customer_pr = c.producer(customer_source_str)

for i in range(100):
    click_m_dict_list = [click_gen.generate() for _ in range(0, 100)]
    click_pr.produce_list(click_m_dict_list)
    #
    customer_m_dict_list = [customer_gen.generate() for _ in range(0, 100)]
    customer_pr.produce_list(customer_m_dict_list)

click_pr.close()
customer_pr.close()


Uptime: 43.974s, State size: 1369.87 KB, Source offsets: {'clicks': {0: 19000}, 'customers': {0: 19000}}, Sink outputs: 16208

'customers'

Uptime: 44.173s, State size: 1392.99 KB, Source offsets: {'clicks': {0: 20000}, 'customers': {0: 20000}}, Sink outputs: 16530

When we see that Kafi Streams has completely read the source topics (=the source offsets have both reached `10000`), you can produce more data at will.

Or stop the Kafi Streams processing thread like so:

In [8]:
stop()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...


Uptime: 54.238s, State size: 1146.50 KB, Source offsets: {'clicks': {0: 20000}, 'customers': {0: 20000}}, Sink outputs: 16530

INFO:kafi.streams.streams:...done.


[]

Let's check whether the sink topic has the same number of messages as displayed by the Kafi Streams progress indicator (`Sink outputs`):

In [9]:
c.l(sink_str)

{'joined': 16530}

And, last but not least, let's have a look at the last ten messages in the sink topic:

In [10]:
c.cat(sink_str, last_n=10)

(['joined'], '1786463733456')


Consuming: 0 msg [00:00, ? msg/s]

[{'value': {'customer_id': 96, 'view_time': 82, 'name': 'Christopher Brown'},
  'key': None,
  'headers': None,
  'timestamp': (1, 1786463712939),
  'partition': 0,
  'offset': 16520,
  'topic': 'joined'},
 {'value': {'customer_id': 96, 'view_time': 82, 'name': 'Cassidy Murray'},
  'key': None,
  'headers': None,
  'timestamp': (1, 1786463712939),
  'partition': 0,
  'offset': 16521,
  'topic': 'joined'},
 {'value': {'customer_id': 97, 'view_time': 39, 'name': 'Todd Russell'},
  'key': None,
  'headers': None,
  'timestamp': (1, 1786463712939),
  'partition': 0,
  'offset': 16522,
  'topic': 'joined'},
 {'value': {'customer_id': 97, 'view_time': 39, 'name': 'Rita Williams'},
  'key': None,
  'headers': None,
  'timestamp': (1, 1786463712939),
  'partition': 0,
  'offset': 16523,
  'topic': 'joined'},
 {'value': {'customer_id': 98, 'view_time': 58, 'name': 'Valerie Bell'},
  'key': None,
  'headers': None,
  'timestamp': (1, 1786463712939),
  'partition': 0,
  'offset': 16524,
  'topic'

Sorry, I know that we could have started with just a map or a filter, but to comprehend the power of Kafi Streams, a slightly more involved example might have been nicer still.

So that's it. Congratulations - that was your first Streams topology in action :-)
